# Academic Literature Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches academic literature across all disciplines using OpenAlex and exports structured citation data for literature reviews. Enter search terms, set year filters, and download publication metadata including titles, authors, abstracts, citation counts, venues, DOIs, and open access status.

OpenAlex indexes over 250 million scholarly works across all fields, making it well-suited for anthropology and social science literature reviews where coverage extends beyond any single disciplinary database.

## Key Features

1. **No API Key Required**: Uses the OpenAlex API, which is completely free and open
2. **All Disciplines**: Covers social sciences, humanities, natural sciences, and everything indexed by Crossref and other sources
3. **Year Filtering**: Restrict results to a specific publication year range
4. **Sorting Options**: Sort by citation count, relevance, or publication date
5. **Open Access Detection**: Identifies which articles are freely accessible
6. **Abstract Retrieval**: Reconstructs abstracts from OpenAlex's inverted index format
7. **Configurable Result Count**: Fetch 10 to 500 results with cursor-based pagination
8. **Structured Export**: Download results as CSV or styled Excel

## Workflow

1. **Configure**: Enter search terms, set year range, sort order, and result count
2. **Fetch**: Retrieve publication data from OpenAlex
3. **Review**: Preview results in a table
4. **Export**: Download structured citation data for further analysis

## Applications

This notebook supports literature reviews across anthropology and the social sciences. Use it to map the scholarly landscape around a research topic, identify highly cited works, track publication trends over time, find open access sources, and build structured reading lists for dissertation research or systematic reviews.

## Methodological Positioning

This notebook represents a **computational anthropology** approach to systematic literature engagement. It collects and structures citation data but does not interpret it. Critical evaluation of sources remains the work of the researcher.

**Important**: OpenAlex has broad coverage but is not exhaustive. For biomedical literature specifically, supplement with the PubMed Literature Harvester notebook.

## Target Audience

Designed for anthropologists and qualitative researchers conducting literature reviews, from graduate students mapping a dissertation field to research teams conducting systematic or scoping reviews.

## Technical Approach

The notebook uses the **OpenAlex API** for search and retrieval. OpenAlex is a free, open catalog of the global research system maintained by OurResearch. It supports keyword search, year-range filtering, sort options, and cursor-based pagination for large result sets. All processing runs in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Priem, Jason, Heather Piwowar, and Richard Orr. 2022. "OpenAlex: A Fully-Open Index of Scholarly Works, Authors, Venues, Institutions, and Concepts." arXiv preprint arXiv:2205.01833.

## Setup and Installation

Install required Python packages and import libraries. Run this cell first.

In [ ]:
# Install required packages
!pip install pandas openpyxl ipywidgets requests -q

import re
import unicodedata
import requests
import pandas as pd
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

OPENALEX_URL = 'https://api.openalex.org/works'


def reconstruct_abstract(inverted_index):
    """Reconstruct abstract text from OpenAlex inverted index format."""
    if not inverted_index:
        return ''
    positions = {}
    for word, pos_list in inverted_index.items():
        for pos in pos_list:
            positions[pos] = word
    return ' '.join(positions[k] for k in sorted(positions))


def extract_work(w):
    """Extract structured data from an OpenAlex work record."""
    authors = '; '.join(
        a.get('author', {}).get('display_name', '')
        for a in w.get('authorships', [])[:10]
    )
    source = w.get('primary_location', {}).get('source', {}) or {}
    venue = source.get('display_name', '')
    oa = w.get('open_access', {})

    return {
        'title': normalize_text(w.get('title', '') or ''),
        'authors': normalize_text(authors),
        'year': w.get('publication_year', ''),
        'venue': normalize_text(venue),
        'citations': w.get('cited_by_count', 0),
        'abstract': normalize_text(reconstruct_abstract(w.get('abstract_inverted_index'))),
        'doi': w.get('doi', '') or '',
        'open_access': 'Yes' if oa.get('is_oa') else 'No',
        'oa_url': oa.get('oa_url', '') or '',
        'type': w.get('type_crossref', '') or w.get('type', ''),
        'openalex_id': w.get('id', ''),
    }


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str): return text
    text = unicodedata.normalize('NFKC', text)
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text


def make_slug(query):
    """Create a clean filename slug."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30] or 'search'


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {"Google Colab" if IN_COLAB else "Local Jupyter"}")
print("\U0001f4da Ready to configure your literature search!")

## Search Configuration

Configure your literature search using the interactive controls below. Set search terms, year range, sort order, and result count.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4da Academic Literature Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook searches OpenAlex (250M+ scholarly works across all disciplines) and exports structured citation data.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Search:</strong> Enter keywords for your literature search</li>'
instructions_html += '<li><strong>Filter:</strong> Set year range, sort order, and result count</li>'
instructions_html += '<li><strong>Fetch:</strong> Click to retrieve results (50 per page, ~1 sec per page)</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f50d Search</h3>')

query_input = widgets.Text(
    value='',
    placeholder='Enter search terms (e.g., "design anthropology")',
    description='Keywords:',
    layout=layout,
    style=style
)

filter_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c5 Filters</h3>')

year_start = widgets.IntText(value=2000, description='From year:', style=style)
year_end = widgets.IntText(value=2026, description='To year:', style=style)
year_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Set both to 0 to search all years.</p>')

sort_input = widgets.Dropdown(
    options=[('Most cited', 'cited_by_count:desc'), ('Most recent', 'publication_date:desc'), ('Relevance', 'relevance')],
    value='cited_by_count:desc',
    description='Sort by:',
    layout=layout,
    style=style
)

max_results_input = widgets.IntSlider(value=100, min=10, max=500, step=10, description='Max results:', style=style)
max_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Results are fetched in pages of 50. 100 results takes ~2 seconds.</p>')

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(description='Search Literature', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))
progress_label = widgets.HTML(value='')
out = widgets.Output()


def on_fetch_clicked(_):
    out.clear_output()
    progress_label.value = ''

    query = query_input.value.strip()
    if not query:
        with out:
            print("\u26a0\ufe0f Please enter search terms.")
        return

    max_count = max_results_input.value
    y_start = year_start.value if year_start.value > 0 else None
    y_end = year_end.value if year_end.value > 0 else None
    sort = sort_input.value

    with out:
        print(f"\U0001f4da Searching OpenAlex for: {query}")
        if y_start and y_end:
            print(f"   Year range: {y_start}-{y_end}")
        print(f"   Sort: {sort_input.label}")
        print(f"   Max results: {max_count}")
        print()

    progress_label.value = '<span style="color: #6096BA;">Searching OpenAlex...</span>'

    try:
        all_rows = []
        cursor = '*'
        page = 0

        while len(all_rows) < max_count and cursor:
            params = {
                'search': query,
                'per_page': min(50, max_count - len(all_rows)),
                'cursor': cursor,
                'mailto': 'toolkit@mattartz.me',
            }

            if sort != 'relevance':
                params['sort'] = sort

            if y_start and y_end:
                params['filter'] = f'publication_year:{y_start}-{y_end}'

            r = requests.get(OPENALEX_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()

            works = data.get('results', [])
            if not works:
                break

            for w in works:
                all_rows.append(extract_work(w))

            cursor = data.get('meta', {}).get('next_cursor')
            page += 1
            total_available = data.get('meta', {}).get('count', 0)
            progress_label.value = f'<span style="color: #274C77;">\u2713 {len(all_rows)} of {min(max_count, total_available):,} results fetched...</span>'

            time.sleep(0.1)

        progress_label.value = ''

        if not all_rows:
            with out:
                print("\u26a0\ufe0f No results found. Try different search terms or a broader year range.")
            return

        df = pd.DataFrame(all_rows)

        with out:
            print(f"\u2713 Retrieved {len(df)} publications (of {total_available:,} available)")
            print()

            # Summary stats
            oa_count = len(df[df['open_access'] == 'Yes'])
            print(f"   Open access: {oa_count} ({100*oa_count/len(df):.0f}%)")
            if 'citations' in df.columns:
                print(f"   Citation range: {df["citations"].min()}-{df["citations"].max()} (median: {df["citations"].median():.0f})")
            print()

            print("\U0001f4ca Preview:")
            display(df[['title', 'authors', 'year', 'citations', 'venue']].head(15))

        # Export
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(query)
        base = f'literature_{slug}_{timestamp}'
        fmt = format_input.value

        if fmt == 'xlsx':
            filepath = f'{base}.xlsx'
            df.to_excel(filepath, sheet_name='Literature Results', index=False, engine='openpyxl')
            style_excel(filepath)
        else:
            filepath = f'{base}.csv'
            df.to_csv(filepath, index=False)

        with out:
            print()
            print(f"\u2713 Saved: {filepath} ({len(df)} publications)")
            if IN_COLAB:
                colab_files.download(filepath)

    except Exception as e:
        progress_label.value = ''
        with out:
            print(f"\u274c Error: {type(e).__name__}: {e}")


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    search_header,
    query_input,
    filter_header,
    widgets.HBox([year_start, year_end]),
    year_help,
    sort_input,
    max_results_input,
    max_help,
    export_header,
    format_input,
    fetch_btn,
    progress_label,
    out,
]))